In [0]:
# Databricks Notebook: 03_raw_to_bronze
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import current_timestamp, col, lit, regexp_extract

spark.sql("CREATE SCHEMA IF NOT EXISTS spotipulse_db")
spark.sql("USE SCHEMA spotipulse_db")

# 1. PROCESS KAGGLE FULL LOAD
kaggle_schema = StructType([
    StructField("title", StringType(), True),
    StructField("rank", IntegerType(), True),
    StructField("date", StringType(), True),
    StructField("artist", StringType(), True),
    StructField("url", StringType(), True),
    StructField("region", StringType(), True),
    StructField("chart", StringType(), True),
    StructField("trend", StringType(), True),
    StructField("streams", IntegerType(), True)
])

KAGGLE_FILE = "/Volumes/workspace/spotipulse_db/raw_data/Kaggle/charts.csv.gz"

try:
    df_kaggle_raw = (spark.read.format("csv")
        .option("header", "true")
        .schema(kaggle_schema)
        .load(KAGGLE_FILE)
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
        .withColumn("data_source", lit("kaggle_full"))
    )

    df_kaggle_raw.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("bronze_kaggle_charts")

    print("✔ Table 'bronze_kaggle_charts' created successfully!")
except Exception as e:
    print(f"✖ Kaggle load error: {e}")

# 2. PROCESS KWORB INCREMENTAL LOAD
KWORB_PATH = "/Volumes/workspace/spotipulse_db/raw_data/kworb/*"

try:
    df_kworb_raw = (spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(KWORB_PATH)
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
        .withColumn("data_source", lit("kworb_incremental"))
        .withColumn("chart_date", regexp_extract("source_file", r"(\d{4}-\d{2}-\d{2})_\w+\.csv", 1))
    )

    df_kworb_raw.write.format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("bronze_kworb_charts")

    print("✔ Table 'bronze_kworb_charts' updated successfully!")
except Exception as e:
    print(f"⚠ Kworb pending (Run GitHub Action + Notebook 02 first): {e}")

In [0]:
# Databricks Notebook: 03_raw_to_bronze
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, TimestampType
from pyspark.sql.functions import current_timestamp, col, lit, regexp_extract
from datetime import datetime

spark.sql("CREATE SCHEMA IF NOT EXISTS spotipulse_db")
spark.sql("USE SCHEMA spotipulse_db")

# Parameter Widget for Backfilling
dbutils.widgets.text("process_date", "", "Process Date (YYYY-MM-DD)")
param_date = dbutils.widgets.get("process_date").strip()
today = param_date if param_date else datetime.utcnow().strftime("%Y-%m-%d")

# 1. SETUP PIPELINE LOGS TABLE
spark.sql("""
CREATE TABLE IF NOT EXISTS pipeline_execution_logs (
    layer STRING,
    parameter_processed STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    status STRING,
    rows_affected LONG,
    error_message STRING
) USING DELTA;
""")

def log_execution(layer, param, start_time, end_time, status, rows, error=""):
    log_data = [(layer, param, start_time, end_time, status, int(rows), error)]
    schema = "layer STRING, parameter_processed STRING, start_time TIMESTAMP, end_time TIMESTAMP, status STRING, rows_affected LONG, error_message STRING"
    df_log = spark.createDataFrame(log_data, schema)
    df_log.write.format("delta").mode("append").saveAsTable("pipeline_execution_logs")

# 2. PROCESS KAGGLE FULL LOAD (Run once if table does not exist)
if not spark.catalog.tableExists("bronze_kaggle_charts"):
    start = datetime.now()
    try:
        kaggle_schema = StructType([
            StructField("title", StringType(), True),
            StructField("rank", IntegerType(), True),
            StructField("date", StringType(), True),
            StructField("artist", StringType(), True),
            StructField("url", StringType(), True),
            StructField("region", StringType(), True),
            StructField("chart", StringType(), True),
            StructField("trend", StringType(), True),
            StructField("streams", IntegerType(), True)
        ])

        KAGGLE_FILE = "/Volumes/workspace/spotipulse_db/raw_data/Kaggle/charts.csv.gz"
        
        df_kaggle_raw = (spark.read.format("csv")
            .option("header", "true")
            .schema(kaggle_schema)
            .load(KAGGLE_FILE)
            .withColumn("load_timestamp", current_timestamp())
            .withColumn("source_file", col("_metadata.file_path"))
            .withColumn("data_source", lit("kaggle_full"))
        )

        df_kaggle_raw.write.format("delta") \
            .mode("overwrite") \
            .option("mergeSchema", "true") \
            .saveAsTable("bronze_kaggle_charts")

        count = spark.table("bronze_kaggle_charts").count()
        log_execution("BRONZE_KAGGLE", "FULL_LOAD", start, datetime.now(), "SUCCESS", count)
        print("✔ Table 'bronze_kaggle_charts' created successfully!")
    except Exception as e:
        log_execution("BRONZE_KAGGLE", "FULL_LOAD", start, datetime.now(), "FAILED", 0, str(e))
        print(f"✖ Kaggle load error: {e}")

# 3. PROCESS KWORB INCREMENTAL LOAD (EXPLICIT SCHEMA)
start = datetime.now()
kworb_schema = StructType([
    StructField("pos", StringType(), True),
    StructField("trend", StringType(), True),
    StructField("artist_title", StringType(), True),
    StructField("days", StringType(), True),
    StructField("peak", StringType(), True),
    StructField("streams", StringType(), True),
    StructField("streams_change", StringType(), True),
    StructField("seven_day", StringType(), True),
    StructField("total", StringType(), True),
    StructField("track_id", StringType(), True),
    StructField("region", StringType(), True)
])

KWORB_PATH = f"/Volumes/workspace/spotipulse_db/raw_data/kworb/{today}/*.csv"

try:
    df_kworb_raw = (spark.read.format("csv")
        .option("header", "true")
        .schema(kworb_schema)
        .load(KWORB_PATH)
        .withColumn("load_timestamp", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
        .withColumn("data_source", lit("kworb_incremental"))
        .withColumn("chart_date", lit(today))
    )

    rows_before = spark.table("bronze_kworb_charts").count() if spark.catalog.tableExists("bronze_kworb_charts") else 0

    df_kworb_raw.write.format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("bronze_kworb_charts")

    rows_after = spark.table("bronze_kworb_charts").count()
    inserted = rows_after - rows_before
    
    log_execution("BRONZE_KWORB", today, start, datetime.now(), "SUCCESS", inserted)
    print(f"✔ Table 'bronze_kworb_charts' appended {inserted} rows for {today}!")
except Exception as e:
    log_execution("BRONZE_KWORB", today, start, datetime.now(), "FAILED", 0, str(e))
    print(f"⚠ Kworb pending for {today}: {e}")